# 🛒 Conversational RAG Support Agent with Hybrid Search & Memory

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY", "")

### 1. Document Ingestion & Chunking

In [7]:
from pathlib import Path
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

data_dir = Path("./data")
raw_docs = []

for file_path in data_dir.glob("*.md"):
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
        raw_docs.append(
            Document(
                page_content=content,
                metadata={"source": file_path.name}
            )
        )
print('Raw documents: ', raw_docs)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " "]
)

chunks = text_splitter.split_documents(raw_docs)
print(f"Loaded {len(raw_docs)} files and created {len(chunks)} chunks.")

Raw documents:  [Document(metadata={'source': 'warranty_rules.md'}, page_content='# Warranty & Replacement Terms\n\n## 1. Coverage Duration & Terms\n- All electronic products purchased from our official store carry a **1-Year Limited Manufacturer Warranty** against hardware defects and workmanship failures.\n- Refurbished items carry a **90-Day Limited Warranty**.\n- Warranty coverage is non-transferable and applies only to the original purchaser with valid order confirmation.\n\n## 2. Exclusions\n- Does not cover accidental damage, liquid spills, unauthorized disassembly, normal wear and tear, or cosmetic scratches.\n\n## 3. Claiming a Replacement (RMA Process)\n- To initiate a warranty claim, submit a request with proof of purchase and a short video/photo demonstrating the failure.\n- If approved, an Return Merchandise Authorization (RMA) label is generated.\n- Replacement units are shipped within 2 business days of defective unit receipt at our service facility.\n'), Document(metada

### 2. Hybrid Retrieval Engine (BM25 + ChromaDB + Reciprocal Rank Fusion)

In [8]:
import chromadb
from chromadb.utils import embedding_functions
from rank_bm25 import BM25Okapi

# 1. Sparse BM25 Keyword Search Index
corpus_tokens = [doc.page_content.lower().split() for doc in chunks]
bm25 = BM25Okapi(corpus_tokens)

# 2. Dense Vector Store (ChromaDB with local ONNX embeddings)
chroma_client = chromadb.EphemeralClient()
embedding_fn = embedding_functions.DefaultEmbeddingFunction()
collection = chroma_client.create_collection(
    name="policy_collection",
    embedding_function=embedding_fn
)

collection.add(
    documents=[doc.page_content for doc in chunks],
    metadatas=[doc.metadata for doc in chunks],
    ids=[f"doc_{i}" for i in range(len(chunks))]
)

def hybrid_search(query: str, top_k: int = 3, k_rrf: int = 60) -> list:
    """Combines BM25 sparse search and ChromaDB dense vector search using Reciprocal Rank Fusion (RRF)."""
    query_tokens = query.lower().split()
    
    # --- Sparse BM25 Ranking ---
    bm25_scores = bm25.get_scores(query_tokens)
    bm25_top_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:10]
    
    # --- Dense Vector Ranking ---
    vector_results = collection.query(query_texts=[query], n_results=10)
    vector_ids = vector_results["ids"][0]
    
    # --- Reciprocal Rank Fusion (RRF) ---
    rrf_scores = {}
    
    # Score BM25 ranks
    for rank, idx in enumerate(bm25_top_indices):
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_rrf + rank + 1))
        
    # Score Vector ranks
    for rank, doc_id in enumerate(vector_ids):
        idx = int(doc_id.split("_")[1])
        rrf_scores[idx] = rrf_scores.get(idx, 0.0) + (1.0 / (k_rrf + rank + 1))
        
    # Sort by fused score
    reranked_indices = sorted(rrf_scores.keys(), key=lambda i: rrf_scores[i], reverse=True)[:top_k]
    
    results = []
    for idx in reranked_indices:
        results.append({
            "content": chunks[idx].page_content,
            "source": chunks[idx].metadata["source"],
            "rrf_score": rrf_scores[idx]
        })
    return results

### 3. RAG Tool & Agent Setup

In [9]:
from langchain_core.tools import tool
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

@tool
def search_company_policies(query: str) -> str:
    """Searches enterprise policy documents (returns, shipping, support, warranty) using Hybrid BM25 + Vector Search."""
    retrieved = hybrid_search(query, top_k=3)
    if not retrieved:
        return "No relevant policy documents found."
        
    formatted_chunks = []
    for item in retrieved:
        source_file = item["source"]
        content_text = item["content"]
        formatted_chunks.append(f"[Source: {source_file}]\n{content_text}")
    return "\n\n---\n\n".join(formatted_chunks)

llm = ChatGroq(model="llama-3.3-70b-versatile")

system_prompt = """You are an expert E-Commerce Customer Support & Returns Agent.

STRICT GROUNDING & CITATION RULES:
1. Always use search_company_policies to retrieve policy documentation before answering questions.
2. Base your answers STRICTLY on the retrieved policy context.
3. ALWAYS cite your source inline using explicit brackets like [Source: return_policy.md].
4. If the retrieved context does NOT contain enough information to answer the question, state: "I don't have enough information in the policy documents to answer that."
5. Politely decline out-of-bounds requests (e.g. personal finance, coding, internal stock prices).
"""

agent = create_agent(
    model=llm,
    tools=[search_company_policies],
    system_prompt=system_prompt,
    checkpointer=MemorySaver()
)

### 4. Stateful Multi-Turn Conversation Demonstration

In [10]:
import uuid

# Create a persistent session thread_id to demonstrate multi-turn conversational memory
thread_id = f"customer_session_{uuid.uuid4().hex[:6]}"
config = {"configurable": {"thread_id": thread_id}}

def chat_with_agent(user_message: str):
    print(f"\nUSER: {user_message}")
    inputs = {"messages": [{"role": "user", "content": user_message}]}
    response = agent.invoke(inputs, config=config)
    final_msg = response["messages"][-1]
    print("AGENT:")
    final_msg.pretty_print()

# --- Turn 1: Specific query about electronics ---
chat_with_agent("Can I return my open-box headphones after 20 days?")

# --- Turn 2: Follow-up question relying on memory ---
chat_with_agent("What about clothing?")

# --- Turn 3: Out-of-bounds / Unsupported query ---
chat_with_agent("What is the company's internal stock price forecast?")


USER: Can I return my open-box headphones after 20 days?
AGENT:
================================== Ai Message ==================================

Yes, you can return your open-box headphones after 20 days [Source: return_policy.md]. According to our return policy, open-box electronics have a 30-day return window from the date of delivery [Source: return_policy.md]. Since it has only been 20 days, you are still within the return window. However, please note that a flat 15% restocking fee will apply unless the item arrived defective or damaged [Source: return_policy.md].

USER: What about clothing?
AGENT:
================================== Ai Message ==================================

You can return clothing within 60 days from the date of delivery [Source: return_policy.md]. Please ensure the item is in its original condition with all tags and packaging intact [Source: return_policy.md].

USER: What is the company's internal stock price forecast?
AGENT:
===============================